### LiDAR building classification with PointNet

In [ ]:
import laspy
import laspy.lasreader
import numpy as np
import rasterio
from pyproj import Transformer

laz_backend = None

try:
    import lazrs
    laspy.lasreader.lazrs = lazrs
    laz_backend = laspy.LazBackend.LazrsParallel
except ImportError:
    pass


def stream_laz_with_raster_labels(laz_path, tif_path, chunk_size=1_000_000, building_threshold=100):
    if laz_backend is None:
        raise RuntimeError("No LAZ backend found. Install lazrs.")

    with rasterio.open(tif_path) as src:
        tif_data = src.read(1)
        transform = src.transform
        target_crs = src.crs

    transformer = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    with laspy.open(laz_path, laz_backend=laz_backend) as reader:
        for chunk in reader.chunk_iterator(chunk_size):
            x = np.asarray(chunk.x)
            y = np.asarray(chunk.y)

            x_t, y_t = transformer.transform(x, y)

            row, col = rasterio.transform.rowcol(transform, x_t, y_t)
            row = np.clip(row, 0, tif_data.shape[0] - 1)
            col = np.clip(col, 0, tif_data.shape[1] - 1)

            label = (tif_data[row, col] >= building_threshold).astype(np.uint8)

            yield {
                "x": x_t,
                "y": y_t,
                "z": np.asarray(chunk.z),
                "intensity": np.asarray(chunk.intensity),
                "return_num": np.asarray(chunk.return_number),
                "num_returns": np.asarray(chunk.number_of_returns),
                "label": label
            }


### Tiling helpers: split the area into 10 m × 10 m cells and write .npz files

In [ ]:
import os
import numpy as np

TILE_SIZE = 10.0
TILE_DIR = "tiles"

os.makedirs(TILE_DIR, exist_ok=True)


def get_tile_ids(x, y, tile_size=TILE_SIZE):
    tile_x = np.floor(x / tile_size).astype(np.int32)
    tile_y = np.floor(y / tile_size).astype(np.int32)
    return tile_x, tile_y


def append_to_tile(tile_path, new_data):
    if os.path.exists(tile_path):
        existing = np.load(tile_path)
        merged = {k: np.concatenate([existing[k], new_data[k]]) for k in new_data}
    else:
        merged = new_data
    np.savez_compressed(tile_path, **merged)


In [ ]:
def tile_streamed_chunk(chunk):
    x = np.asarray(chunk["x"])
    y = np.asarray(chunk["y"])
    z = np.asarray(chunk["z"])
    intensity = np.asarray(chunk["intensity"])
    return_num = np.asarray(chunk["return_num"])
    num_returns = np.asarray(chunk["num_returns"])
    label = np.asarray(chunk["label"])

    tile_x = np.floor(x / TILE_SIZE).astype(np.int32)
    tile_y = np.floor(y / TILE_SIZE).astype(np.int32)

    unique_tiles = np.unique(np.stack([tile_x, tile_y], axis=1), axis=0)

    for tx, ty in unique_tiles:
        mask = (tile_x == tx) & (tile_y == ty)
        tile_data = {
            "x": x[mask], "y": y[mask], "z": z[mask],
            "intensity": intensity[mask],
            "return_num": return_num[mask],
            "num_returns": num_returns[mask],
            "label": label[mask]
        }
        tile_path = os.path.join(TILE_DIR, f"tile_{tx}_{ty}.npz")
        append_to_tile(tile_path, tile_data)


In [ ]:
LAZ_FILE = "209400MANIKPUR_209399JAYAPUR.laz"
TIF_FILE = "209400MANIKPUR_209399JAYAPUR_ORTHO.tif"
CHUNK_SIZE = 1_000_000

for chunk in stream_laz_with_raster_labels(
    LAZ_FILE, TIF_FILE, chunk_size=CHUNK_SIZE
):
    tile_streamed_chunk(chunk)

print("STEP 2 COMPLETED")


### Extract per-point features from each tile and save as parquet

In [ ]:
import os
import numpy as np
import pandas as pd

FEATURE_DIR = "features"
os.makedirs(FEATURE_DIR, exist_ok=True)


In [ ]:
def core_mask(x, y, tx, ty):
    x_min = tx * TILE_SIZE
    x_max = x_min + TILE_SIZE
    y_min = ty * TILE_SIZE
    y_max = y_min + TILE_SIZE

    return (
        (x >= x_min) & (x < x_max) &
        (y >= y_min) & (y < y_max)
    )


In [ ]:
def extract_features_from_tile(tile_path):
    data = np.load(tile_path)

    x = data["x"]
    y = data["y"]
    z = data["z"]
    intensity = data["intensity"]
    return_num = data["return_num"]
    num_returns = data["num_returns"]
    label = data["label"]

    if len(z) == 0:
        return pd.DataFrame()

    name = os.path.basename(tile_path).replace("tile_", "").replace(".npz", "")
    tx, ty = map(int, name.split("_"))

    mask = core_mask(x, y, tx, ty)
    if mask.sum() == 0:
        return pd.DataFrame()

    return pd.DataFrame({
        "x": x[mask], "y": y[mask], "z": z[mask],
        "intensity": intensity[mask],
        "return_num": return_num[mask],
        "num_returns": num_returns[mask],
        "label": label[mask],
        "tile_x": tx,
        "tile_y": ty
    })


In [ ]:
import os

tile_files = [os.path.join(TILE_DIR, f) for f in os.listdir(TILE_DIR) if f.endswith(".npz")]

print(f"Processing {len(tile_files)} tiles...")

for tile_path in tile_files:
    df = extract_features_from_tile(tile_path)
    if df is None or df.empty:
        continue
    out_name = os.path.basename(tile_path).replace(".npz", ".parquet")
    df.to_parquet(os.path.join(FEATURE_DIR, out_name), index=False)

print("Feature extraction done")


### Load all features and set up for training

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

dfs = []
for f in os.listdir(FEATURE_DIR):
    if f.endswith(".parquet"):
        dfs.append(pd.read_parquet(os.path.join(FEATURE_DIR, f)))

data = pd.concat(dfs, ignore_index=True)

print("Total samples:", len(data))
print(data["label"].value_counts())


In [ ]:
BLOCK_SIZE = 5

data["block_x"] = data["tile_x"] // BLOCK_SIZE
data["block_y"] = data["tile_y"] // BLOCK_SIZE
data["block_id"] = data["block_x"].astype(str) + "_" + data["block_y"].astype(str)

unique_blocks = data["block_id"].unique()
train_blocks, test_blocks = train_test_split(unique_blocks, test_size=0.2, random_state=42)

train_data = data[data["block_id"].isin(train_blocks)]
test_data = data[data["block_id"].isin(test_blocks)]

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))


In [ ]:
train_points = train_data[["x", "y", "z"]].values
train_labels = train_data["label"].values

test_points = test_data[["x", "y", "z"]].values
test_labels = test_data["label"].values


def create_blocks(points, labels, block_size=1024):
    blocks = []
    block_labels = []
    for i in range(0, len(points), block_size):
        p = points[i:i + block_size]
        l = labels[i:i + block_size]
        if len(p) < block_size:
            continue
        blocks.append(p)
        block_labels.append(l)
    return np.array(blocks), np.array(block_labels)


POINT_BLOCK_SIZE = 1024
X_train, y_train = create_blocks(train_points, train_labels, POINT_BLOCK_SIZE)
X_test, y_test = create_blocks(test_points, test_labels, POINT_BLOCK_SIZE)

print("Train blocks:", X_train.shape, y_train.shape)
print("Test blocks:", X_test.shape, y_test.shape)


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### Define PointNetSeg and run the training loop

In [ ]:
class PointNetSeg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)

        self.conv4 = nn.Conv1d(1088, 512, 1)
        self.conv5 = nn.Conv1d(512, 256, 1)
        self.conv6 = nn.Conv1d(256, num_classes, 1)

    def forward(self, x):
        B, N, _ = x.shape
        x = x.transpose(2, 1)

        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x1))
        x3 = F.relu(self.conv3(x2))

        global_feat = torch.max(x3, 2, keepdim=True)[0]
        global_feat = global_feat.repeat(1, 1, N)

        x = torch.cat([x1, global_feat], dim=1)

        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.conv6(x)

        x = x.transpose(2, 1)
        return x


train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

epochs = 10
model = PointNetSeg(num_classes=2)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Starting training on device:", device)
print(f"Total batches per epoch: {len(dataloader)}\n")

for epoch in range(epochs):
    model.train()

    epoch_loss = 0.0
    for batch_points, batch_labels in dataloader:
        batch_points = batch_points.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        outputs = model(batch_points)
        loss = criterion(outputs.reshape(-1, 2), batch_labels.reshape(-1))

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.6f}")


model = model.to("cpu")
print("Model moved to CPU for saving")


### Save the trained weights

In [ ]:
MODEL_PATH = "pointnet_model.pt"
torch.save(model.state_dict(), MODEL_PATH)
print("Model saved to", MODEL_PATH)